# aw_03_c — Stages C1/C2: instruct reference baselines (G4 anchor)

**Protocol**: §4 reference baselines, §5.1 C1/C2. No training — two eval runs of
Qwen/Qwen3-8B (post-trained), pinned revision `b968826d…`.

- **C1**: zero-shot, no opener seed (the instruct model writes its own think block).
- **C2**: few-shot k=3 with FROZEN exemplars drawn from train families only
  (`scripts/build_fewshot_exemplars.py`) — leakage gate preserved.

Suites, verifier, and the greedy decoding profile are identical to A1/base evals;
only the conditioning profile (`evaluation:` config block) differs, per amendment v1.1.


In [ ]:
# @title common header
import os

from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
os.environ["WANDB_API_KEY"] = userdata.get('WANDB_API_KEY')
os.environ["GITHUB_TOKEN"] = userdata.get('GITHUB_TOKEN')

!git clone https://{os.environ["GITHUB_TOKEN"]}@github.com/m97j/axiom-world.git
%cd axiom-world
!pip install -e . -r requirements/colab-g4.lock.txt


In [ ]:
# @title a_c_data — frozen suites + frozen few-shot exemplars
!python scripts/build_eval_suites.py --episodes-per-suite 300
!python scripts/build_training_data.py
!python scripts/build_fewshot_exemplars.py \
  --sft-file data/train/playworld_sft.jsonl \
  --output data/eval_suites/fewshot_exemplars.json \
  --count 3


In [ ]:
# @title c_c1_eval — instruct zero-shot
!python scripts/run_evaluation.py \
  --config configs/experiments/eval_playworld_c1.yaml \
  --max-new-tokens 1024 --batch-size 100 \
  --hf-sync-repo m97j/aw-runs-a1


In [ ]:
# @title d_c2_eval — instruct few-shot (k=3, frozen exemplars)
!python scripts/run_evaluation.py \
  --config configs/experiments/eval_playworld_c2.yaml \
  --max-new-tokens 1024 --batch-size 100 \
  --hf-sync-repo m97j/aw-runs-a1


In [ ]:
# @title f_c_analysis — anchor comparisons
RUN_ID_a1   = "20260801-063425--eval-playworld--s42--3bf440"  # A1 eval run
RUN_ID_c1   = ""  # <- fill from c_c1_eval output ("eval run: ...")
RUN_ID_c2   = ""  # <- fill from d_c2_eval output

!python scripts/fetch_run.py --repo m97j/aw-runs-a1 --run-id {RUN_ID_a1} --kind eval

!python scripts/run_analysis.py \
  --run-a runs/{RUN_ID_a1} --label-a a1-sft \
  --run-b runs/{RUN_ID_c1} --label-b qwen3-8b-instruct-0shot \
  --output runs/{RUN_ID_a1}/analysis_vs_c1.json --hf-sync-repo m97j/aw-runs-a1

!python scripts/run_analysis.py \
  --run-a runs/{RUN_ID_a1} --label-a a1-sft \
  --run-b runs/{RUN_ID_c2} --label-b qwen3-8b-instruct-3shot \
  --output runs/{RUN_ID_a1}/analysis_vs_c2.json --hf-sync-repo m97j/aw-runs-a1


## Gate checklist (G4)
- [ ] C1/C2 summaries persisted to the Hub
- [ ] A1 vs C1 / A1 vs C2 analysis synced
- [ ] Reference anchors recorded — headline tables may now cite C1/C2
